# 04 — Diferenciação, ACF e PACF

No notebook anterior, vimos que a série original ainda não apresentou evidência suficiente de estacionariedade.

Agora vamos aplicar uma transformação importante em séries temporais: a diferenciação.

Neste notebook, vamos:

- criar a primeira diferença da série;
- comparar a série original com a série diferenciada;
- aplicar novamente o teste ADF;
- visualizar ACF e PACF;
- preparar a intuição dos parâmetros do ARIMA.

### 04.01 Imports e caminhos

In [10]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import plotly.graph_objects as go

from statsmodels.tsa.stattools import adfuller, acf, pacf

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    aplicar_layout_padrao,
    grafico_linha_padrao
)

caminho_serie_diaria = raiz_projeto / "outputs" / "tabelas" / "serie_diaria_bike.csv"
caminho_outputs_tabelas = raiz_projeto / "outputs" / "tabelas"

print("Série diária:")
print(caminho_serie_diaria)

Série diária:
c:\GitHub\ALURA\series-temporais-bikes\outputs\tabelas\serie_diaria_bike.csv


### 04.02 Carregar série diária

In [11]:
serie_diaria = pd.read_csv(caminho_serie_diaria)

serie_diaria["data_hora"] = pd.to_datetime(serie_diaria["data_hora"])

serie_diaria = serie_diaria.sort_values("data_hora").reset_index(drop=True)

print(f"Linhas: {serie_diaria.shape[0]}")
print(f"Colunas: {serie_diaria.shape[1]}")

serie_diaria.head()

Linhas: 456
Colunas: 2


,data_hora,demanda
0,2011-01-01,985
1,2011-01-02,801
2,2011-01-03,1349
3,2011-01-04,1562
4,2011-01-05,1600


## Retomando a série original

Antes de diferenciar, vamos visualizar novamente a série diária.

A diferenciação será aplicada sobre a coluna `demanda`.

### 04.03 Visualizar série original

In [12]:
fig = grafico_linha_padrao(
    df=serie_diaria,
    x="data_hora",
    y="demanda",
    titulo="Série original — demanda diária",
    labels={
        "data_hora": "Data",
        "demanda": "Demanda diária"
    },
    altura=500
)

fig.show()

## Primeira diferença

A diferenciação calcula a mudança entre um valor e o valor anterior.

Em vez de olhar apenas para a pergunta:

> qual foi a demanda?

passamos a olhar também para:

> quanto a demanda mudou em relação ao período anterior?

### 04.04 Criar série diferenciada

In [13]:
serie_diaria["demanda_diff"] = serie_diaria["demanda"].diff()

serie_diaria[["data_hora", "demanda", "demanda_diff"]].head(10)

,data_hora,demanda,demanda_diff
0,2011-01-01,985,NaN
1,2011-01-02,801,-184.0
2,2011-01-03,1349,548.0
3,2011-01-04,1562,213.0
4,2011-01-05,1600,38.0
5,2011-01-06,1606,6.0
6,2011-01-07,1510,-96.0
7,2011-01-08,959,-551.0
8,2011-01-09,822,-137.0
9,2011-01-10,1321,499.0


### 04.05 Remover valor nulo da diferença

In [14]:
serie_diferenciada = serie_diaria.dropna(subset=["demanda_diff"]).copy()

print(f"Linhas da série original: {serie_diaria.shape[0]}")
print(f"Linhas da série diferenciada: {serie_diferenciada.shape[0]}")

serie_diferenciada.head()

Linhas da série original: 456
Linhas da série diferenciada: 455


,data_hora,demanda,demanda_diff
1,2011-01-02,801,-184.0
2,2011-01-03,1349,548.0
3,2011-01-04,1562,213.0
4,2011-01-05,1600,38.0
5,2011-01-06,1606,6.0


### 04.06 Visualizar série diferenciada

In [15]:
fig = grafico_linha_padrao(
    df=serie_diferenciada,
    x="data_hora",
    y="demanda_diff",
    titulo="Série diferenciada — variação da demanda diária",
    labels={
        "data_hora": "Data",
        "demanda_diff": "Variação da demanda"
    },
    altura=500
)

fig.show()

## Teste ADF na série diferenciada

Agora vamos aplicar o teste ADF novamente.

A ideia é verificar se a diferenciação deixou a série com comportamento mais estável.

### 04.07 Função para teste ADF

In [16]:
def teste_adf(serie):
    resultado = adfuller(serie.dropna())

    tabela_resultado = pd.DataFrame({
        "métrica": [
            "estatística_adf",
            "p_valor",
            "lags_utilizados",
            "observações"
        ],
        "valor": [
            resultado[0],
            resultado[1],
            resultado[2],
            resultado[3]
        ]
    })

    valores_criticos = pd.DataFrame({
        "métrica": [f"valor_crítico_{chave}" for chave in resultado[4].keys()],
        "valor": list(resultado[4].values())
    })

    return pd.concat([tabela_resultado, valores_criticos], ignore_index=True)

### 04.08 Aplicar ADF na série diferenciada

In [17]:
resultado_adf_diferenciada = teste_adf(serie_diferenciada["demanda_diff"])

resultado_adf_diferenciada

,métrica,valor
0,estatística_adf,-1.175157e+01
1,p_valor,1.205722e-21
2,lags_utilizados,8.000000e+00
3,observações,4.460000e+02
4,valor_crítico_1%,-3.445097e+00
5,valor_crítico_5%,-2.868042e+00
6,valor_crítico_10%,-2.570233e+00


### 04.09 Interpretar p-valor

In [18]:
p_valor = resultado_adf_diferenciada.loc[
    resultado_adf_diferenciada["métrica"] == "p_valor",
    "valor"
].iloc[0]

print(f"p-valor: {p_valor:.4f}")

if p_valor <= 0.05:
    print("Resultado: há evidência de estacionariedade após a diferenciação.")
    print("Isso sugere que d = 1 pode ser um bom ponto de partida para o ARIMA.")
else:
    print("Resultado: ainda não há evidência suficiente de estacionariedade.")
    print("Poderíamos avaliar novas transformações, mas vamos seguir com cuidado na modelagem.")

p-valor: 0.0000
Resultado: há evidência de estacionariedade após a diferenciação.
Isso sugere que d = 1 pode ser um bom ponto de partida para o ARIMA.


## ACF e PACF

Depois de diferenciar a série, vamos observar sua memória temporal.

A ACF mostra a autocorrelação total da série com seus valores passados.

A PACF tenta isolar o efeito direto de cada defasagem.

### 04.10 Função para gráfico de lags

In [ ]:
def grafico_lags(valores, titulo, nome_eixo_y):
    lags = np.arange(len(valores))
    limite_confianca = 1.96 / np.sqrt(len(serie_diferenciada))

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=lags,
            y=valores,
            marker_color=CORES["azul_principal"],
            name=nome_eixo_y
        )
    )

    fig.add_hline(
        y=0,
        line_width=1,
        line_color=CORES["cinza_suave"]
    )

    fig.add_hline(
        y=limite_confianca,
        line_dash="dash",
        line_color=CORES["azul_claro"]
    )

    fig.add_hline(
        y=-limite_confianca,
        line_dash="dash",
        line_color=CORES["azul_claro"]
    )

    fig = aplicar_layout_padrao(
        fig,
        titulo=titulo,
        altura=450,
        legenda_horizontal=False
    )

    fig.update_xaxes(title="Lag")
    fig.update_yaxes(title=nome_eixo_y)

    return fig

### 04.11 Calcular ACF

In [ ]:
valores_acf = acf(
    serie_diferenciada["demanda_diff"],
    nlags=30
)

fig = grafico_lags(
    valores=valores_acf,
    titulo="ACF — Autocorrelação da série diferenciada",
    nome_eixo_y="Autocorrelação"
)

fig.show()

### 04.12 Calcular PACF

In [ ]:
valores_pacf = pacf(
    serie_diferenciada["demanda_diff"],
    nlags=30,
    method="ywm"
)

fig = grafico_lags(
    valores=valores_pacf,
    titulo="PACF — Autocorrelação parcial da série diferenciada",
    nome_eixo_y="Autocorrelação parcial"
)

fig.show()

## Leitura para o ARIMA

A diferenciação ajuda a pensar no parâmetro `d`.

A ACF e a PACF ajudam a levantar valores iniciais para `p` e `q`.

Essa leitura não garante o melhor modelo, mas dá um ponto de partida para a modelagem.

### 04.13 Registrar parâmetros iniciais

In [ ]:
parametros_iniciais = pd.DataFrame({
    "parâmetro": ["p", "d", "q"],
    "ideia": [
        "memória dos valores passados",
        "número de diferenciações aplicadas",
        "memória dos erros passados"
    ],
    "ponto_de_partida": [
        "avaliar pela PACF",
        1,
        "avaliar pela ACF"
    ]
})

parametros_iniciais

### 04. 14 Salvar resultados

In [ ]:
caminho_serie_diferenciada = caminho_outputs_tabelas / "serie_diferenciada_bike.csv"
caminho_adf_diferenciada = caminho_outputs_tabelas / "resultado_adf_diferenciada.csv"
caminho_parametros_iniciais = caminho_outputs_tabelas / "parametros_iniciais_arima.csv"

serie_diferenciada.to_csv(
    caminho_serie_diferenciada,
    index=False,
    encoding="utf-8-sig"
)

resultado_adf_diferenciada.to_csv(
    caminho_adf_diferenciada,
    index=False,
    encoding="utf-8-sig"
)

parametros_iniciais.to_csv(
    caminho_parametros_iniciais,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos:")
print("-", caminho_serie_diferenciada)
print("-", caminho_adf_diferenciada)
print("-", caminho_parametros_iniciais)

## Próximo passo

Neste notebook, transformamos a série com diferenciação e começamos a ler sua memória temporal com ACF e PACF.

No próximo notebook, vamos usar essa preparação para treinar o primeiro modelo ARIMA.